In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        _tok = ""
        try:
            from google.colab import userdata
            _tok = userdata.get("GH_TOKEN") or ""
        except Exception:
            _tok = ""
        if not _tok:
            print("WARNING: no 'GH_TOKEN' Colab secret found; cloning this PRIVATE repo will fail.\n"
                  "Add a GitHub token (repo scope) via the key icon (Secrets) as 'GH_TOKEN', then re-run.")
        _url = (f"https://{_tok}@github.com/{_slug}.git" if _tok
                else f"https://github.com/{_slug}.git")
        subprocess.run(["git", "clone", "--depth", "1", _url, str(_root)], check=True)
        subprocess.run(["git", "-C", str(_root), "remote", "set-url", "origin",
                        f"https://github.com/{_slug}.git"])  # keep the token out of the saved remote
    os.chdir(_root / "07-application-agent-framework/long-running-durable/long-running-agentic/long-running-agents-gcp/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 03 · HITL + saga — practice
Reference: `notebooks/solutions/ex3_hitl_saga.py`.

In [ ]:
import sys, os, json, warnings
warnings.filterwarnings("ignore")
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))          # repo root when run from notebooks/
sys.path[:0] = [os.path.join(ROOT, "src"), os.path.join(ROOT, "notebooks")]

def show_journal(run):
    print(f"run {run.run_id}  status={run.status.value}  version={run.version}  steps={run.usage.steps}  tokens={run.usage.tokens}  cost=${run.usage.cost_usd:.4f}")
    for s in run.journal:
        out = json.dumps(s.output, default=str)[:70] if s.output is not None else (s.error or "")
        print(f"  [{s.index}] {s.kind.value:<6} {s.status.value:<7} {s.name:<18} key={s.idempotency_key or '-':<20} {out}")

In [ ]:
import hmac
from lragents.core import *
from lragents.practice_checks import check_approve, check_saga_next

## Exercise 1 — `approve`
Implement `approve(loop, run_id, token, approved, approver, comment="")`:
1. `acquire_lease`; if status is not `WAITING_HUMAN`, return the run (idempotent);
2. compare the token with `hmac.compare_digest` — raise on mismatch;
3. append a `HUMAN` step (`DONE`) with the decision; clear `waiting_on`; set `RUNNING`;
4. if approved, append a `TOOL` step with status **`STARTED`**, the *stored* tool name/args and key `f"{run_id}:{index}"` — do **not** call the LLM;
5. `loop.store.save(run)`, `loop._enqueue_next(run)`, release the lease in `finally`.

In [ ]:
def approve(loop, run_id, token, approved, approver, comment=""):
    run = loop.store.acquire_lease(run_id, loop.worker_id, loop.lease_ttl_s)
    try:
        # TODO
        raise NotImplementedError
    finally:
        loop.store.release_lease(run_id, loop.worker_id)

In [ ]:
print(check_approve(approve))

## Exercise 2 — saga state machine
`saga_next(order, saga)` returns which step runs next and in what phase, given
`saga = {"phase": "forward"|"compensating", "cursor": int, "comp_cursor": int|None}`.
Return `("__done__", "succeeded")` when all forward steps completed, `("__done__", "failed")` when compensation walked past the first step.

In [ ]:
def saga_next(order, saga):
    # TODO
    raise NotImplementedError

In [ ]:
print(check_saga_next(saga_next))

## Exercise 3 — design questions
1. Why must the approved tool call be journaled *before* the loop is woken, rather than passed in the wake-up payload?
2. The approval endpoint is public behind IAP. List three checks the handler must do before mutating the run.
3. A compensation ("refund") fails 5 times. What state should the saga end in, who is notified, and what must NOT happen?

In [ ]:
answers = '''
1.
2.
3.
'''

In [ ]:
import inspect
from solutions import ex3_hitl_saga as ref
print(inspect.getsource(ref.approve)); print(inspect.getsource(ref.saga_next))